In [17]:
from pathlib import Path
import pandas as pd

import numpy as np
import cv2
from skimage.graph import route_through_array
from scipy.ndimage import distance_transform_edt


In [18]:

def index_segmentations_df(root_dir):
    root_dir = Path(root_dir)
    images_root = root_dir / "images"
    masks_root = root_dir / "masks"

    records = []

    for scan_dir in images_root.iterdir():
        if not scan_dir.is_dir():
            continue

        scan_name = scan_dir.name

        for class_dir in scan_dir.iterdir():
            if not class_dir.is_dir():
                continue

            class_name = class_dir.name

            for img_path in class_dir.iterdir():
                if img_path.suffix.lower() not in [".png", ".jpg", ".tif", ".tiff"]:
                    continue

                mask_path = masks_root / scan_name / class_name / img_path.name

                if not mask_path.exists():
                    raise FileNotFoundError(f"Missing mask for {img_path}")

                records.append({
                    "scan": scan_name,
                    "class": class_name,
                    "image_path": img_path,
                    "mask_path": mask_path,
                })

    df = pd.DataFrame(records)

    # Optional: sort for reproducibility
    df = df.sort_values(
        by=["scan", "class", "image_path"],
        ignore_index=True
    )

    return df


In [19]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from pathlib import Path

def show_row_visuals(
    df,
    idx,
    columns,
    cmap_mask="gray"
):
    """
    Show original image + multiple masks/images side by side.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing image/mask columns
    idx : int
        Row index in the DataFrame
    columns : list[str]
        Columns to visualize (can be paths or numpy arrays)
        e.g. ["mask_path", "true_mask"]
    cmap_mask : str
        Colormap for mask-like visuals
    """
    row = df.iloc[idx]

    # Always load original image
    image = np.array(Image.open(row.image_path).convert("RGB"))

    n_plots = 1 + len(columns)
    fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 5))

    if n_plots == 1:
        axes = [axes]

    # --- Original image ---
    axes[0].imshow(image)
    axes[0].set_title("Original image")
    axes[0].axis("off")

    # --- Additional columns ---
    for ax, col in zip(axes[1:], columns):
        value = row[col]

        # Case 1: numpy array (e.g. computed mask)
        if isinstance(value, np.ndarray):
            ax.imshow(value, cmap=cmap_mask)

        # Case 2: path to image/mask
        elif isinstance(value, (str, Path)):
            arr = np.array(Image.open(value))
            if arr.ndim == 2:
                ax.imshow(arr, cmap=cmap_mask)
            else:
                ax.imshow(arr)

        else:
            raise TypeError(f"Unsupported type for column '{col}': {type(value)}")

        ax.set_title(col)
        ax.axis("off")

    plt.suptitle(
        f"Scan: {row.scan} | Class: {row['class']}",
        fontsize=12
    )
    plt.tight_layout()
    plt.show()


In [20]:

import numpy as np
from PIL import Image
import cv2

def compute_brown_red_mask_hsv(
    image_path,
    mask_path=None,
    hue_range=(5, 35),     # brown/red hues
    sat_thresh=0.2,       # remove white background
):
    """
    Extract brown/red regions using HSV color space.

    Optionally intersects with an existing mask.
    """
    image = np.array(Image.open(image_path).convert("RGB"))

    # Convert to HSV (OpenCV uses H: [0,179])
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    h = hsv[:, :, 0].astype(np.float32) * 2.0   # → [0, 360]
    s = hsv[:, :, 1].astype(np.float32) / 255.0

    # Brown/red condition
    color_mask = (
        (h >= hue_range[0]) &
        (h <= hue_range[1]) &
        (s >= sat_thresh)
    )

    # Optional: intersect with original annotation
    if mask_path is not None:
        orig_mask = np.array(Image.open(mask_path)) > 0
        color_mask = color_mask & orig_mask

    return color_mask.astype(np.uint8)


In [21]:
def compute_brown_red_mask_hsv_adaptive(
    image_path,
    mask_path=None,
    hue_range=(5, 35),
    min_sat=0.08,
    morph_kernel=5
):
    image = np.array(Image.open(image_path).convert("RGB"))
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)

    h = hsv[:, :, 0].astype(np.float32) * 2.0
    s = hsv[:, :, 1].astype(np.float32) / 255.0

    color_mask = (
        (h >= hue_range[0]) &
        (h <= hue_range[1]) &
        (s >= min_sat)
    )

    color_mask = color_mask.astype(np.uint8)

    # Cleanup
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (morph_kernel, morph_kernel))
    color_mask = cv2.morphologyEx(color_mask, cv2.MORPH_OPEN, kernel)
    color_mask = cv2.morphologyEx(color_mask, cv2.MORPH_CLOSE, kernel)

    if mask_path is not None:
        orig_mask = np.array(Image.open(mask_path)) > 0
        color_mask &= orig_mask

    return color_mask.astype(np.uint8)


In [22]:
# def connect_components_geodesic(
#     binary_mask,
#     gray_image,
#     max_connections=5,
# ):
#     """
#     Connect disconnected components using shortest paths
#     through low-intensity image regions.

#     Parameters
#     ----------
#     binary_mask : (H, W) uint8
#     gray_image  : (H, W) uint8
#     max_connections : int
#         Max number of component connections

#     Returns
#     -------
#     connected_mask : (H, W) uint8
#     """

#     connected_mask = binary_mask.copy()

#     # Label connected components
#     num_labels, labels = cv2.connectedComponents(binary_mask)

#     if num_labels <= 2:
#         return connected_mask  # already connected

#     # Cost image: prefer dark pixels
#     cost = gray_image.astype(np.float32)
#     cost = cost / cost.max()
#     cost += 1e-3  # avoid zero-cost

#     # Find centroids
#     centroids = []
#     for label in range(1, num_labels):
#         ys, xs = np.where(labels == label)
#         centroids.append((int(np.mean(ys)), int(np.mean(xs))))

#     # Connect closest components iteratively
#     connections = 0
#     for i in range(len(centroids) - 1):
#         if connections >= max_connections:
#             break

#         start = centroids[i]
#         end = centroids[i + 1]

#         path, _ = route_through_array(
#             cost,
#             start,
#             end,
#             fully_connected=True
#         )

#         for y, x in path:
#             connected_mask[y, x] = 1

#         connections += 1

#     return connected_mask.astype(np.uint8)


In [23]:
def compute_similarity_cost_rgb(image):
    img = image.astype(np.float32)

    dx = np.linalg.norm(
        img[:, 1:] - img[:, :-1],
        axis=-1
    )
    dx = np.pad(dx, ((0, 0), (1, 0)))

    dy = np.linalg.norm(
        img[1:, :] - img[:-1, :],
        axis=-1
    )
    dy = np.pad(dy, ((1, 0), (0, 0)))

    cost = dx + dy
    cost = cost / (cost.max() + 1e-6)
    cost += 1e-3

    return cost

# def compute_similarity_cost_rgb(image, white_penalty=10.0):
#     """
#     Cost is low for:
#       - similar neighboring pixels
#       - dark pixels

#     Cost is high for:
#       - edges
#       - bright / white background
#     """
#     img = image.astype(np.float32)

#     # --- Similarity term ---
#     dx = np.linalg.norm(img[:, 1:] - img[:, :-1], axis=-1)
#     dx = np.pad(dx, ((0, 0), (1, 0)))

#     dy = np.linalg.norm(img[1:, :] - img[:-1, :], axis=-1)
#     dy = np.pad(dy, ((1, 0), (0, 0)))

#     similarity = dx + dy
#     similarity = similarity / (similarity.max() + 1e-6)

#     # --- Brightness penalty ---
#     lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
#     L = lab[:, :, 0].astype(np.float32) / 255.0  # [0,1]

#     # Penalize bright pixels strongly
#     brightness_penalty = 1.0 + white_penalty * (L ** 2)

#     # --- Final cost ---
#     cost = similarity * brightness_penalty
#     cost += 1e-3  # avoid zero cost

#     return cost



In [ ]:
def connect_components_geodesic_similarity(
    binary_mask,
    image,
    max_connections=5,
    restrict_to_mask=True,
):
    """
    Connect components by minimizing pixel-to-pixel appearance differences.
    """

    connected_mask = binary_mask.copy()

    num_labels, labels = cv2.connectedComponents(binary_mask)
    if num_labels <= 2:
        return connected_mask

    # --- ALWAYS define cost ---
    # if image.ndim == 3:
    cost = compute_similarity_cost_rgb(image)
    # else:
    #     cost = compute_similarity_cost_gray(image)

    # Optional: strongly discourage leaving original mask
    if restrict_to_mask:
        penalty = cost.max() * 10
        cost[~binary_mask.astype(bool)] += penalty

    # Compute centroids
    centroids = []
    for label in range(1, num_labels):
        ys, xs = np.where(labels == label)
        centroids.append((int(np.mean(ys)), int(np.mean(xs))))

    # Connect components sequentially
    connections = 0
    for i in range(len(centroids) - 1):
        if connections >= max_connections:
            break

        start = centroids[i]
        end = centroids[i + 1]

        path, _ = route_through_array(
            cost,
            start,
            end,
            fully_connected=True
        )

        for y, x in path:
            connected_mask[y, x] = 1

        connections += 1

    return connected_mask.astype(np.uint8)


In [25]:
def compute_mask_otsu_inside_annotation(
    image_path,
    mask_path,
    connect_components=True,
):
    image = np.array(Image.open(image_path).convert("RGB"))
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    orig_mask = np.array(Image.open(mask_path)) > 0

    roi = gray.copy()
    roi[~orig_mask] = 255

    _, thresh = cv2.threshold(
        roi, 0, 1, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    kernel = np.ones((3, 3), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)

    mask = (thresh & orig_mask).astype(np.uint8)

    if connect_components:
        mask = connect_components_geodesic_similarity(mask, gray)

    return mask


In [26]:
def add_true_mask_column(df):
    df = df.copy()

    df["true_mask"] = df.apply(
        # lambda row: compute_brown_red_mask_hsv_adaptive(
        #     row.image_path,
        #     row.mask_path
        # ),
        lambda row: compute_mask_otsu_inside_annotation(
            row.image_path,
            row.mask_path
        ),
        axis=1
    )

    return df


Read annotation data

In [27]:
df = index_segmentations_df(
    r"C:\Users\chris\Desktop\University\Thesis\AnnotationsData\Segmentations"
)

# Only include the whole cell annotation for now, later we can do a custom soma segmentation
df = df[df['class'] == 'MG_whole']
print(len(df), "annotation pairs found")
df.head()

print(df.columns)



df = add_true_mask_column(df)
show_row_visuals(df, 1, columns = ["mask_path", 'true_mask'])


1009 annotation pairs found
Index(['scan', 'class', 'image_path', 'mask_path'], dtype='object')


UnboundLocalError: cannot access local variable 'cost' where it is not associated with a value